# RAG (2020)
---
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
RAG = Retrieval-Augmented Generation

RAG — это гибридная модель для генерации текста, предложенная Facebook AI, которая совмещает непараметрический механизм извлечения (retriever) с параметрической генеративной моделью (generator) для получения ответов на запросы. Модель обучена end-to-end, что позволяет ей динамически обращаться к внешней базе знаний в процессе генерации.

**Контекст:**
Большие языковые модели (LLMs) демонстрируют впечатляющие способности в генерации связного и грамматически корректного текста. Однако у них есть существенные недостатки:
1.  **Галлюцинации:** LLMs могут генерировать фактологически неверную информацию, уверенно представляя её как истину. Это связано с тем, что их "знания" хранятся в параметрах модели и могут быть неточными или устаревшими.
2.  **Устаревшие знания:** Параметры моделей фиксируются на момент их обучения, и они не имеют доступа к новой информации, появляющейся после их `pre-training`.
3.  **Отсутствие атрибуции:** LLMs не могут указать источник информации, которую они используют, что затрудняет проверку фактов и доверие к сгенерированному контенту.

**Идея метода:**
Решение RAG заключается в том, чтобы преодолеть ограничения параметрических LLMs, объединив их с непараметрической системой поиска информации. Вместо того, чтобы полагаться исключительно на свои внутренние, потенциально устаревшие или неточные знания, модель RAG сначала извлекает релевантные документы из большого корпуса текста, а затем использует эти документы как контекст для генерации ответа. Ключевая новизна состоит в **совместном (end-to-end) обучении** retriever и generator, позволяя retriever'у научиться извлекать документы, наиболее полезные для generator'а.

**Постановка задачи:**
RAG решает задачу **Open-Domain Generative Question Answering**, где модель должна не только найти ответ в заданной коллекции документов, но и сформулировать его в связном виде, используя информацию из извлеченных пассажей.

**Существующие альтернативные методы (до RAG):**
1.  **Чисто генеративные модели:** Такие как GPT-2 (2019) или T5 (2019). Они генерируют ответы исключительно на основе своих параметрических знаний.
    *   *Отличие от RAG:* Не имеют механизма обращения к внешней базе знаний, подвержены галлюцинациям и не могут предоставить источники.
2.  **Extractive QA модели:** Например, fine-tuned BERT (2018) на SQuAD. Эти модели находят точный участок текста (span) из данного контекста, который является ответом.
    *   *Отличие от RAG:* Требуют, чтобы ответ буквально присутствовал в тексте. Не способны синтезировать новый ответ, обобщать или перефразировать информацию, как RAG.
3.  **Dense Passage Retrieval (DPR) (2020):** Является мощным методом для поиска релевантных документов, но сам по себе не генерирует ответы, а только извлекает документы.
    *   *Отличие от RAG:* DPR — это *часть* или *компонент* RAG, но не полноценная система генерации ответов. RAG использует такой retriever и добавляет к нему generator.
4.  **Ранние гибридные/Knowledge-Grounded модели:** Были попытки использовать структурированные базы знаний или более простые методы поиска для информирования генерации, но они часто были ограничены качеством retriever'а или сложностью интеграции.
    *   *Отличие от RAG:* RAG предлагает элегантное и эффективное решение благодаря использованию мощных neural retriever'ов и generator'ов, обученных end-to-end.

**Архитектура модели:**
Архитектура RAG состоит из двух основных компонентов, работающих совместно:
1.  **Retriever (Извлекатель):** Отвечает за поиск релевантных документов из большой коллекции (например, статей Wikipedia).
    *   Представляет собой **Dense Retriever**, похожий на DPR (2020). Он состоит из двух независимых `encoder` моделей:
        *   **Query Encoder:** Принимает входной запрос (`query`) и преобразует его в `dense embedding`.
        *   **Passage Encoder:** Преобразует каждый документ (или `passage` из документа) в коллекции в `dense embedding`.
    *   Релевантность между запросом и документом измеряется через `dot product` их `embeddings`.
    *   Коллекция документов предварительно индексируется (например, с помощью FAISS) для быстрого поиска ближайших соседей.
2.  **Generator (Генератор):** Модель типа `sequence-to-sequence` (например, BART (2019) или T5 (2019)), которая принимает на вход исходный запрос и **извлеченные retriever'ом документы**, а затем генерирует ответ.
    *   Генератор использует архитектуру `encoder-decoder`. `Encoder` обрабатывает конкатенацию запроса и релевантных документов, а `decoder` генерирует ответ на основе этого закодированного контекста.

**Алгоритм обучения:**
Обучение RAG происходит end-to-end, что является ключевым для его эффективности.
1.  **Инициализация:** Retriever инициализируется `pre-trained` DPR (2020) model (Query Encoder и Passage Encoder). Generator инициализируется `pre-trained` `sequence-to-sequence` моделью, такой как BART (2019).
2.  **Шаги обучения:**
    *   Для каждого обучающего примера (запрос, правильный ответ):
        *   **`Retrieval`:** `Query Encoder` получает запрос, ищет в базе знаний `k` наиболее релевантных `passages` с помощью `dot product` и `Passage Encoder`.
        *   **`Generation`:** Для каждого из `k` извлеченных `passages` генератор вычисляет вероятность генерации правильного ответа, обусловливая её запросом и данным `passage`.
        *   **`Loss Calculation`:** Функция потерь — это отрицательный логарифм маргинальной вероятности. Вместо того чтобы полагаться на *один* лучший `passage`, RAG **маргинализирует** по всем извлеченным `passages`. Это означает, что он суммирует вероятности генерации правильного ответа, обусловленные каждым из `k` `passages`. Таким образом, генератор учится давать правильный ответ, даже если retriever извлек несколько потенциально полезных `passages`.
        *   **`Backpropagation`:** Градиенты от функции потерь распространяются через generator и retriever, позволяя обеим моделям обучаться совместно. Retriever учится извлекать документы, которые наиболее полезны для генератора для получения правильного ответа.

**Алгоритм инференса:**
1.  **Входной запрос:** Пользователь вводит запрос.
2.  **`Retrieval`:** `Query Encoder` преобразует запрос в `embedding`. Этот `embedding` используется для поиска `k` наиболее релевантных `passages` в предварительно индексированной базе знаний (например, с помощью FAISS).
3.  **`Generation`:** Каждый из `k` извлеченных `passages` вместе с исходным запросом подается на вход `encoder`'а генератора. `Decoder` генератора затем генерирует ответ.
4.  **Маргинализация и финальный ответ:** Для генерации финального ответа RAG опять же использует маргинализацию. Он рассматривает вероятность каждого токена ответа, обусловленную каждым из `k` извлеченных `passages`. Это позволяет генератору выбирать наиболее вероятные токены, учитывая всю доступную релевантную информацию, а не полагаться на один лучший `passage`. Результатом является `coherent` и `factually grounded` ответ.

**Результаты:**
RAG модель была протестирована на нескольких задачах Open-Domain QA, включая Natural Questions и WebQuestions.
*   **Улучшение F1 и EM:** RAG значительно улучшил метрики `Exact Match (EM)` и `F1-score` по сравнению с `state-of-the-art` чисто генеративными моделями (например, `fine-tuned` BART (2019) или T5 (2019)), особенно на сложных вопросах, требующих внешней информации. Например, на Natural Questions RAG (`RAG-Sequence`) достиг 44.5% EM против 34.0% у BART-large.
*   **Снижение галлюцинаций:** Модель демонстрировала значительно меньшее количество галлюцинаций, поскольку она `grounded` свои ответы на реальных документах из базы знаний.
*   **Атрибуция:** Хотя RAG напрямую не выводит источники, сама его архитектура позволяет отследить, какие документы были использованы для генерации ответа, что значительно повышает прозрачность и доверие к модели.
*   **Конкуренция с Extractive QA:** RAG показал результаты, сравнимые или даже превосходящие некоторые `state-of-the-art` `extractive` `QA` системы на ряде бенчмарков, при этом предлагая более гибкий, генеративный формат ответов.

## 📝 Критический анализ

```markdown
# RAG (2020)
---
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
RAG = Retrieval-Augmented Generation

RAG — гибридная модель от Facebook AI, объединяющая извлечение информации с генерацией текста для ответов на запросы. Обучена end-to-end, что позволяет динамически обращаться к внешней базе знаний.

**Контекст:**
Большие языковые модели (LLMs) могут генерировать текст, но страдают от галлюцинаций, устаревших знаний и отсутствия атрибуции. 

**Идея:**
RAG преодолевает ограничения LLMs, сочетая их с системой поиска информации. Модель извлекает релевантные документы и использует их для генерации ответов. Ключевая новизна — **совместное (end-to-end) обучение** retriever и generator.

**Постановка задачи:**
Решается задача **Open-Domain Generative Question Answering**.

**Альтернативы:**
1. **Генеративные модели:** GPT-2 (2019), T5 (2019) — не обращаются к внешним данным.
2. **Extractive QA:** BERT (2018) — требует наличия ответа в тексте.
3. **Dense Passage Retrieval (DPR) (2020):** извлекает документы, но не генерирует ответы.
4. **Ранние гибридные модели:** ограничены качеством retriever'а.

**Архитектура:**
1. **Retriever:** Ищет релевантные документы, используя **Dense Retriever**.
   - **Query Encoder** и **Passage Encoder** преобразуют запросы и документы в `embeddings`.
   - Релевантность измеряется через `dot product`.
2. **Generator:** Генерирует ответ, используя `sequence-to-sequence` модель (например, BART (2019)).

<img src="img/img.png" width=500>

**Алгоритм обучения:**
1. **Инициализация:** Retriever инициализируется DPR, Generator — BART.
2. **Шаги обучения:**
   - **Retrieval:** Извлечение `k` релевантных `passages`.
   - **Generation:** Генерация ответа с учетом извлеченных `passages`.
   - **Loss Calculation:** Маргинализация вероятностей генерации.
   - **Backpropagation:** Совместное обучение retriever и generator.

**Алгоритм инференса:**
1. **Входной запрос:** Преобразуется в `embedding`.
2. **Retrieval:** Поиск `k` релевантных `passages`.
3. **Generation:** Генерация ответа с учетом всех `passages`.
4. **Маргинализация:** Выбор наиболее вероятных токенов.

**Результаты:**
- **Улучшение F1 и EM:** На Natural Questions RAG достиг 44.5% EM против 34.0% у BART-large.
- **Снижение галлюцинаций:** Ответы основаны на реальных документах.
- **Атрибуция:** Возможность отследить использованные документы.
- **Конкуренция с Extractive QA:** Сравнимые или превосходящие результаты на бенчмарках.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример использования Retrieval-Augmented Generation (RAG) с использованием библиотеки Hugging Face Transformers.
# Этот пример иллюстрирует, как можно использовать RAG для генерации ответов на вопросы с использованием внешней базы знаний.

from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration

# Инициализация токенизатора, извлекателя и генератора RAG
tokenizer = RagTokenizer.from_pretrained("facebook/rag-sequence-nq")
retriever = RagRetriever.from_pretrained("facebook/rag-sequence-nq", index_name="exact", use_dummy_dataset=True)
model = RagSequenceForGeneration.from_pretrained("facebook/rag-sequence-nq")

# Входной запрос
query = "What is the capital of France?"

# Токенизация запроса
input_ids = tokenizer(query, return_tensors="pt").input_ids

# Извлечение релевантных документов
# Retriever использует dense embeddings для поиска релевантных документов
retrieved_docs = retriever(input_ids)

# Генерация ответа
# Генератор использует извлеченные документы для генерации ответа
outputs = model.generate(input_ids=input_ids, context_input_ids=retrieved_docs['context_input_ids'])

# Декодирование и вывод сгенерированного ответа
generated_answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(f"Generated Answer: {generated_answer}")

# Комментарии:
# 1. RAG состоит из двух основных компонентов: Retriever и Generator.
# 2. Retriever извлекает релевантные документы из базы знаний, используя dense embeddings.
# 3. Generator использует извлеченные документы для генерации ответа на основе запроса.
# 4. Модель обучена end-to-end, что позволяет Retriever и Generator работать совместно.
# 5. В отличие от чисто генеративных моделей, RAG использует внешнюю информацию, что снижает вероятность галлюцинаций.
```

Этот пример демонстрирует использование модели RAG для генерации ответа на вопрос с использованием внешней базы знаний. В отличие от чисто генеративных моделей, RAG сначала извлекает релевантные документы, а затем использует их для генерации ответа, что позволяет снизить вероятность галлюцинаций и улучшить фактологическую точность.